In [1]:
import paho.mqtt.client as mqtt
import time
from flask import Flask, render_template, request, redirect
import firebase_admin
from firebase_admin import credentials, firestore
from firebase_admin import db

# Initialize Flask app
app = Flask(__name__)

# Initialize Firebase credentials and Firestore database
cred = credentials.Certificate("key.json")
firebase_admin.initialize_app(cred, {'databaseURL': 'https://iotproject-db999-default-rtdb.firebaseio.com/'})

ref = db.reference('/')

temp_data = None
dht = None

In [2]:
def on_message(client, userdata, message):
    global temp_data
    msg = message.payload.decode("utf-8")
    temp_data = msg.split('|')
    temp_data = {
        'temp': temp_data[0],
        'hum': temp_data[1],
        'weather':temp_data[2]
    }


client = mqtt.Client("subscriber")
client.connect("192.168.19.1", 1883)
client.subscribe("temp", 0)
client.on_message = on_message

# while True:
#     client.loop_start()  
#     time.sleep(0.1)  
#     client.loop_stop()  
#     if temp_data:
#         print(temp_data)
#     else:
#         print("LOL")



# Route to render the HTML page with LED data
@app.route('/')
def index():
    # Retrieve LED data from Firebase
    leds_ref = ref.child('home').get()
    leds = leds_ref
    while True:
        client.loop_start()  
        time.sleep(0.1)  
        client.loop_stop()   
        if temp_data:
            dht = temp_data
        else:
            dht = {
            'temp': "25",
            'hum': "45",
            'weather':"Sunny"
            }

        led_data = []
        for position, info in leds_ref.items():
            led_data.append({'position': position, 'state': info['led']})
        return render_template('index.html', led_data=led_data, dht=dht)

# Route to handle LED actions
@app.route('/action', methods=['POST'])
def action():
    position = request.form['position']
    action = request.form['action']
    
    # Update LED state in Firebase
    led_data = 'on' if action == 'on' else 'off'
    ref.child('home/'+position+'/led').set(led_data)
    
    # Redirect back to the main route
    return redirect('/')

if __name__ == '__main__':
    app.run()


 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)
127.0.0.1 - - [10/May/2024 11:14:17] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 11:14:17] "GET /static/Assets/Design/css/bootstrap.min.css HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 11:14:28] "POST /action HTTP/1.1" 302 -
127.0.0.1 - - [10/May/2024 11:14:29] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 11:14:37] "POST /action HTTP/1.1" 302 -
127.0.0.1 - - [10/May/2024 11:14:38] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 11:14:41] "POST /action HTTP/1.1" 302 -
127.0.0.1 - - [10/May/2024 11:14:42] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 11:14:45] "POST /action HTTP/1.1" 302 -
127.0.0.1 - - [10/May/2024 11:14:46] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 11:14:46] "GET /favicon.ico HTTP/1.1" 404 -


In [29]:
from flask import Flask, render_template, request, redirect
import firebase_admin
from firebase_admin import credentials, firestore
from firebase_admin import db

# Initialize Flask app
app = Flask(__name__)

# Initialize Firebase credentials and Firestore database
cred = credentials.Certificate("key.json")
firebase_admin.initialize_app(cred, {'databaseURL': 'https://iotproject-db999-default-rtdb.firebaseio.com/'})

ref = db.reference('/')

# Route to render the HTML page with LED data
@app.route('/')
def index():
    # Retrieve LED data from Firebase
    leds_ref = ref.child('home').get()
    dht = ref.child('DHT').get()
    leds = leds_ref
    led_data = []
    for position, info in leds_ref.items():
        led_data.append({'position': position, 'state': info['led']})
    return render_template('index.html', led_data=led_data, dht=dht)

# Route to handle LED actions
@app.route('/action', methods=['POST'])
def action():
    position = request.form['position']
    action = request.form['action']
    
    # Update LED state in Firebase
    led_data = 'on' if action == 'on' else 'off'
    ref.child('home/'+position+'/led').set(led_data)
    
    # Redirect back to the main route
    return redirect('/')

if __name__ == '__main__':
    app.run()

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)
127.0.0.1 - - [10/May/2024 00:16:23] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 00:16:23] "GET /static/Assets/Design/css/bootstrap.min.css HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2024 00:16:23] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [10/May/2024 00:16:23] "GET /static/Assets/Design/css/blog.css HTTP/1.1" 200 -


In [48]:
from flask import Flask, render_template, request, redirect

# Initialize Flask app
app = Flask(__name__)

led_data = {
  "office": {"led": "on"},
  "room1": {"led": "off"}
}



# Route to render the HTML page with LED data
@app.route('/')
def index():
    return render_template('index.html', led_data=led_data)

# Route to handle LED actions
@app.route('/action', methods=['POST'])
def action():
    position = request.form['position']
    action = request.form['action']

    led_data[position]['led'] = 'on' if action == 'on' else 'off'
    
    # Redirect back to the main route
    return redirect('/')

if __name__ == '__main__':
    app.run()

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)
127.0.0.1 - - [09/May/2024 17:11:43] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [09/May/2024 17:11:52] "GET /static/Assets/Design/css/bootstrap.min.css.map HTTP/1.1" 404 -


27.7